In [1]:
# pip install --upgrade sendgrid

In [2]:
# pip install --upgrade cryptography

In [3]:
# pip install --user openai-agents

In [1]:
import os
import asyncio
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
from dotenv import load_dotenv

In [2]:
from agents import Agent, Runner, trace, function_tool

In [3]:
from openai.types.responses import ResponseTextDeltaEvent

In [4]:
load_dotenv(override=True)

True

In [5]:
# def send_test_email():
#     sg = sendgrid.SendGridAPIClient(api_key=os.getenv("SENDGRID_API_KEY"))
#     from_email = Email("wetechfin@gmail.com")
#     to_email = To("debmalyamondal63@gmail.com")
#     content = Content("text/plain", "This is a the test email body")
#     mail = Mail(from_email, to_email, "test email", content).get()
#     response = sg.client.mail.send.post(request_body=mail)
#     print(response.status_code)

In [6]:
# send_test_email()

In [7]:
instruction1 = "You are a sales agent working for AutAI, \
an AI automation agency company that provides automated solution for small and medium business. \
You write professional, serious cold emails"

instruction2 = "You are a humorous, engaging sales agent working for AutAI, \
an AI automation agency company that provides automated solution for small and medium businesses. \
You write witty, engaging cold emails that are likely to get a response."

instruction3 = "You are a busy sales agent working for AutAI, \
an AI automation agency company that provides automated solution for small and medium businesses. \
You write concise, to the point cold emails"



In [8]:
sales_agent1 = Agent(
    name="Professional sales agent",
    instructions=instruction1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Engaging sales agent",
    instructions=instruction2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Busy sales agent",
    instructions=instruction3,
    model="gpt-4o-mini"
)


In [9]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, flush=True, end="")

Subject: Unlock Efficiency with AI Automation Solutions

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I represent AutAI, an AI automation agency dedicated to helping small and medium businesses like yours streamline operations and enhance productivity.

In today’s fast-paced market, efficiency is key. Our custom automation solutions can help you reduce operational costs, minimize errors, and maximize your team's potential—all while saving valuable time.

Here are a few ways we can assist:

1. **Workflow Automation**: Streamline repetitive tasks to allow your team to focus on what truly matters.
2. **Data Analysis**: Quickly analyze large datasets for informed decision-making without the manual labor.
3. **Customer Engagement**: Implement AI-driven tools that enhance customer interactions and drive satisfaction.

I would love to discuss how we can tailor our services to meet your specific needs and help you achieve your business goals. Would 

In [10]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

In [11]:
message = "Write a cold email"

with trace("Parellel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
    outputs = [result.final_output for result in results]
    
    emails = "cold sales emails: \n\n" + "\n\nEmail: \n\n".join(outputs)
    
    best = await Runner.run(sales_picker, emails)
    
    print(f"Best sales email: \n{best.final_output}")

Best sales email: 
Subject: Let’s Automate Your Business Like It’s 2099 🚀

Hey [Recipient's Name],

Hope this email finds you conquering the day (or at least planning your next coffee break)! ☕️ 

I'm reaching out from AutAI, where we specialize in turning mundane tasks into magical automated solutions—kind of like how a microwave turns leftovers into “gourmet” meals (with a bit of imagination, of course).

Picture this: Your business humming along effortlessly while you sit back, relax, and finally get to that *very important* Netflix binge. 🎬 How, you ask? With our automated solutions tailored for small and medium businesses, we can help you save time, reduce errors, and improve efficiency faster than you can say “I need another coffee!”

Let’s grab a virtual coffee (or a well-deserved nap) and chat about how we can sprinkle some automation magic into your business. Ready to say goodbye to manual headaches and hello to smoother operations? 

Looking forward to hearing from you!

Best